# 01 - Exploratory data analysis

**Goal.** A first-time CT user should finish this notebook understanding what the data looks
like, what a clot looks like, and where clots occur along the body. It also freezes the
study-level splits (seeded) that the fine-tuning notebook reuses.

## CT scans for ML engineers (read once)

- **A CT scan is a 3D grayscale volume**, shape roughly `(slices, 512, 512)`. The closest
  computer-vision analogy is a short video clip whose frames are cross-sections of the body
  from the neck down to the belly. One frame is a **slice**.
- **Pixel values are Hounsfield Units (HU)**, a calibrated physical density scale. Air is about
  -1000, fat about -100, water 0, soft tissue about +40, contrast-filled blood +200 to +400,
  bone about +1000. The same number means the same tissue on any scanner.
- **Windowing** means clipping to an HU range and rescaling to [0, 1] for display or for a
  model. Different windows reveal different tissue.
- **Voxel spacing** is the physical size of one voxel in mm. It varies by scanner.
- **DICOM** is the medical image file format: one file per slice with a metadata header.
  Hierarchy: **Study** (one exam = one sample) -> **Series** (one 3D reconstruction; exactly
  one per study here) -> **SOP Instance** (one slice file).
- **Pulmonary embolism (PE)** is a blood clot blocking arteries in the lungs. **CTPA** is a
  chest CT taken right after injecting contrast dye, so blood in the lung arteries is bright.
- **A clot looks like a dark spot inside a bright vessel** (a *filling defect*). In CV terms:
  small, low-contrast dark objects inside thin bright tubes, somewhere in a large 3D volume.
- **Label limitation.** Labels say *which slices* contain a clot, not *where in the slice*.

Glossary for this notebook:

- **Slice**: one 2D frame of the 3D volume, like one frame of a video. **Axial** slices are
  the native frames (seen from the feet); **coronal** and **sagittal** views are re-slices
  from the front and from the side.
- **HU** (Hounsfield units): calibrated density. Air -1000, fat -100, water 0, soft tissue
  +40, contrast-filled blood +200 to +400, bone +1000.
- **Window**: an HU range mapped to black..white for display. The lung window shows air-filled
  detail, the soft-tissue window organs, the vessel window the contrast-filled arteries.
- **CTPA**: chest CT taken right after injecting contrast dye, so blood in the lung arteries is bright.
- **Filling defect**: a dark spot inside a bright vessel where the dye could not flow because a
  clot is in the way. This is what a PE looks like.
- **Sub-labels** (PE studies only): *central* (clot in the big arteries near the heart) vs
  peripheral; *RV/LV ratio >= 1* (the right heart chamber is enlarged, a sign of strain from a
  big clot); *chronic* vs acute (old vs fresh clot); *left-/right-sided*.
- **Look-alikes**: *flow artifact* (dye mixing unevenly) and *true filling defect, not PE*
  (something else blocking the dye). They fool humans and models alike.

In [ ]:
import os, sys, subprocess

# Python 3.13 removed 'imp' but IPython's autoreload still imports it
if sys.version_info >= (3, 13) and "imp" not in sys.modules:
    import importlib, types
    imp_mock = types.ModuleType("imp"); imp_mock.reload = importlib.reload; sys.modules["imp"] = imp_mock

BRANCH = "main"
REPO = "https://github.com/SattamAltwaim/rsna-pe-ct.git"
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = os.path.exists("/content") and not ON_KAGGLE
DEST = "/kaggle/working/rsna-pe-ct" if ON_KAGGLE else "/content/rsna-pe-ct"

if ON_COLAB or ON_KAGGLE:                            # hosted: clone or fast-forward, install
    if not os.path.exists(DEST):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO, DEST], check=True)
    else:
        subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", DEST], check=True)
    if DEST not in sys.path:
        sys.path.insert(0, DEST)
    if ON_COLAB:
        from google.colab import drive; drive.mount("/content/drive")
else:                                                # local fallback (Mac)
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "pe_ct")) and parent not in sys.path:
        sys.path.insert(0, parent)

%load_ext autoreload
%autoreload 2
import pe_ct
print("pe_ct", pe_ct.__version__, "| commit:",
      subprocess.run(["git", "-C", DEST if os.path.exists(DEST) else "..", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip())

## Configuration

In [ ]:
import os
from pe_ct.config import default_config
from pe_ct import viz

NOTEBOOK = "01_eda"
cfg = default_config()                       # Kaggle: mounted dataset + /kaggle/working; Colab: S3 + Drive; local: ./data
SMOKE = os.environ.get("PE_CT_SMOKE") == "1" # tiny sizes for a quick end-to-end check
if SMOKE:
    cfg.n_eda, cfg.n_dev, cfg.n_test_sample, cfg.shard_size = 4, 60, 20, 10

cfg.ensure_dirs()
if cfg.source == "kaggle":                   # outputs of earlier notebooks attached as inputs
    from pe_ct import colab
    from pe_ct.config import describe_kaggle_inputs
    print("competition data:", cfg.kaggle_input or "NOT FOUND - attach it via Add Input -> Competitions")
    print("/kaggle/input contains:"); print(describe_kaggle_inputs())
    print("linked earlier outputs:", colab.link_kaggle_inputs(cfg))
viz.use_notebook_style()
FIG = cfg.figures_dir
print(cfg)

## Labels, splits and the EDA volumes

The slice-level label file is collapsed to one row per study (three mutually exclusive
groups: negative, PE, indeterminate; indeterminate studies are dropped from the binary
task). A stratified test pool is frozen first and never touched until the final cell of
the fine-tuning notebook; from the rest, the dev subset (with group-stratified folds) and a
class-balanced EDA subset are drawn. All of it is seeded, so every notebook rebuilds the
identical table.

Volumes are read through a small store object, so no cell handles files directly. On
Kaggle the store reads straight from the mounted dataset: metadata comes from the DICOM
headers (fast) and a volume is decoded only when a plot needs it. On Colab the store
reads the shards built by the previous notebook (building any that are missing).

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pe_ct import labels, pipeline, storage, volume

studies, split_table = pipeline.ensure_labels_and_splits(cfg)
slice_index = labels.SliceLabelIndex(pipeline.load_train(cfg))
store = pipeline.eda_volume_store(cfg, studies, split_table, slice_index, pipeline.make_locator(cfg), progress=tqdm)
eda_uids = store.uids()
print(len(studies), "labelled studies |", len(eda_uids), "EDA volumes available |", type(store).__name__)

In [ ]:
from pe_ct import splits

usable = labels.usable_studies(studies)
print(len(usable), "usable studies |", int(usable.y.sum()), "PE positive |", f"{usable.y.mean():.1%} prevalence")
summary = splits.split_summary(split_table)
summary

### Class balance per split

**What you're looking at.** One group of bars per subset; bar height is the share of PE-positive studies (left) and the share of central or heart-straining PE among the positives (right).

**Why it matters.** If stratification worked, every subset has the same mix, so a score on the dev folds transfers to the test sample. A mismatch here would make every later comparison unfair.

**What to look for.** All bars within a panel at about the same height.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
summary["pe_frac"].plot.bar(ax=axes[0], color=viz.COLOR_PE, title="PE prevalence per subset")
summary[["central_pe_frac_of_pe", "rv_lv_gte_1_frac_of_pe"]].plot.bar(ax=axes[1], title="sub-type share among PE studies")
for ax in axes:
    ax.set_ylim(0, 1); ax.set_xlabel("")
fig.tight_layout()
viz.save_fig(fig, FIG, NOTEBOOK, "class_balance_per_split");

In [ ]:
metas = {u: store.load_meta(u) for u in tqdm(eda_uids, desc="metadata")}
acq = pd.DataFrame([{
    "study_uid": u, "y": m["y"], "n_slices": m["shape_zyx"][0],
    "slice_spacing_mm": m["slice_spacing_median_mm"], "inplane_mm": m["spacing_xyz_mm"][0],
    "scanner": m.get("scanner", m["manufacturer"]), "kvp": m["kvp"], "n_missing": m["n_missing_slices_est"],
} for u, m in metas.items()]).set_index("study_uid")
pe_uids = [u for u in eda_uids if metas[u]["y"] == 1]
neg_uids = [u for u in eda_uids if metas[u]["y"] == 0]
print(len(pe_uids), "PE |", len(neg_uids), "negative in the EDA subset")

## A. The labels (all studies)

### Study groups

**What you're looking at.** Three bars: the number of negative, PE-positive and indeterminate studies in the whole label file.

**Why it matters.** This is the class imbalance the classifier has to live with, and the reason we report precision-recall and macro F1 rather than plain accuracy.

**What to look for.** Roughly two negatives per positive; a small indeterminate group that we exclude.

In [ ]:
import matplotlib.pyplot as plt

counts = labels.group_counts(studies)["n_studies"]
fig, ax = plt.subplots(figsize=(6, 3.5))
viz.bar_counts(ax, counts, colors=[viz.COLOR_NEG, viz.COLOR_PE, viz.COLOR_IND], title="study groups")
viz.save_fig(fig, FIG, NOTEBOOK, "study_groups");

### Sub-label prevalence among PE studies

**What you're looking at.** Horizontal bars: the fraction of PE-positive studies carrying each sub-label.

**Why it matters.** Tells us which kinds of PE are common and which are rare; rare kinds need stratified sampling and get their own recall numbers later.

**What to look for.** Acute, one-sided clots dominate; central clots and heart strain are a minority; chronic clots are rare.

In [ ]:
prev = labels.sublabel_prevalence(studies)
fig, ax = plt.subplots(figsize=(7, 3.5))
viz.bar_counts(ax, prev.round(3), horizontal=True, title="share of PE studies with each sub-label")
ax.set_xlim(0, 1)
viz.save_fig(fig, FIG, NOTEBOOK, "sublabel_prevalence");

### Sub-label co-occurrence

**What you're looking at.** A matrix: cell (row, column) is the probability that a PE study with the row label also carries the column label. The diagonal is 1 by construction.

**Why it matters.** The labels are not independent: a clot that is central tends to strain the heart, and a two-sided clot is both left- and right-sided. Any analysis by sub-type has to keep this in mind.

**What to look for.** Bright off-diagonal cells, especially central -> RV/LV >= 1. Mutually exclusive pairs (RV/LV >= 1 vs < 1) are 0.

In [ ]:
co = labels.sublabel_cooccurrence(studies)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(co.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(co))); ax.set_xticklabels(co.columns, rotation=60, ha="right")
ax.set_yticks(range(len(co))); ax.set_yticklabels(co.index)
for i in range(len(co)):
    for j in range(len(co)):
        ax.text(j, i, f"{co.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if co.values[i, j] > 0.6 else "k")
ax.set_title("P(column | row) among PE studies"); fig.colorbar(im, fraction=0.046)
viz.save_fig(fig, FIG, NOTEBOOK, "sublabel_cooccurrence");

### Positive slices per PE study

**What you're looking at.** Histogram of how many slices are labelled positive in each PE study, on a log x-axis.

**Why it matters.** A clot visible on one slice is a tiny object in a volume of hundreds of slices; one visible on two hundred slices is huge. This spread predicts which studies will be hard for a whole-scan embedding.

**What to look for.** A wide, roughly log-normal spread: the median is a few dozen slices, with a long tail in both directions.

In [ ]:
pe_all = studies[studies.group == "pe"]
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(0, np.log10(pe_all.n_pos_slices.max() + 1), 30)
ax.hist(pe_all.n_pos_slices, bins=bins, color=viz.COLOR_PE)
ax.set_xscale("log"); ax.set_xlabel("positive slices per PE study"); ax.set_ylabel("studies")
ax.axvline(pe_all.n_pos_slices.median(), color="k", ls="--", lw=1, label=f"median = {pe_all.n_pos_slices.median():.0f}")
ax.legend()
viz.save_fig(fig, FIG, NOTEBOOK, "positive_slices_per_study");

## B. The scans

### Interactive slice browser

Pick a study, drag the slider to move along the z-axis (top of the scan at 0), change the
window. The thin strip on the right marks the PE-positive slices in red and the orange line
is the slice you are looking at. This is the single most useful way to "get" a CT:
it really is a stack of frames.

In [ ]:
viz.slice_scroller(store.load_volume, pe_uids[:20] + neg_uids[:10], labels_lookup=lambda u, m: m["slice_labels"])

### The same slice in four windows

**What you're looking at.** One axial slice through the lung arteries of a PE study (a slice labelled positive), shown with the full HU range and three clinical windows. Level L is the HU value mapped to mid-gray; width W the HU range spanned from black to white.

**Why it matters.** A model's input preprocessing is exactly such a window. The clot is invisible in some windows and visible in others, which is why the preprocessing must match what the backbone was trained with.

**What to look for.** Full range: almost everything is mid-gray. Lung window: fine airway detail, vessels saturated white. Soft tissue and vessel windows: bright branching arteries in the center with (possibly) a darker spot inside.

In [ ]:
demo_uid = max(pe_uids, key=lambda u: metas[u]["n_pos_slices"])   # the clearest clot: most positive slices
demo_vol, demo_meta = store.load_volume(demo_uid)
demo_z = labels.representative_positive_slice(demo_meta["slice_labels"])
fig = viz.window_comparison(demo_vol[demo_z], title=f"study {demo_uid}, slice {demo_z} (PE positive)")
viz.save_fig(fig, FIG, NOTEBOOK, "windowing_comparison");

### HU histograms

**What you're looking at.** Voxel-value histograms (log y) of a few volumes, with dashed lines at the textbook HU of air, fat, water, soft tissue, contrast-filled blood and bone.

**Why it matters.** Checks that the intensity calibration is right: if the peaks did not line up with the physical tissues, every window and the backbone's preprocessing would be off.

**What to look for.** A huge peak at -1000 (air inside and around the body), a broad bump around 0 to +60 (soft tissue), a shoulder in the +200 to +400 range (contrast in vessels) and a long tail to +1000 and beyond (bone).

In [ ]:
sample_uids = pe_uids[:2] + neg_uids[:2]
vols = [store.load_volume(u)[0] for u in sample_uids]
fig, ax = plt.subplots(figsize=(9, 4))
viz.hu_histogram(ax, vols, labels=[f"{u} (y={metas[u]['y']})" for u in sample_uids])
viz.save_fig(fig, FIG, NOTEBOOK, "hu_histograms");

### Acquisition variability

**What you're looking at.** Four panels over the EDA volumes: number of slices, slice spacing (mm), in-plane pixel size (mm), and the scanner with the PE rate per scanner written above each bar. The manufacturer tag is missing from almost every file in this dataset, so the vendor-specific reconstruction kernel name stands in for it.

**Why it matters.** Raw scans are inconsistent in resolution, which motivates resampling or a model robust to it. The last panel is a shortcut check: if one manufacturer contributed mostly PE cases, a model could learn to recognise the scanner instead of the disease.

**What to look for.** Slice spacing clustering at a few standard values; in-plane size varying with patient size. PE rates per scanner close to the overall prevalence; a scanner far from it is a confounder to watch.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
axes[0].hist(acq.n_slices, bins=30, color=viz.PALETTE[0]); axes[0].set_title("slices per study")
axes[1].hist(acq.slice_spacing_mm, bins=30, color=viz.PALETTE[1]); axes[1].set_title("slice spacing (mm)")
axes[2].hist(acq.inplane_mm, bins=30, color=viz.PALETTE[2]); axes[2].set_title("in-plane pixel size (mm)")
man = acq.groupby("scanner").agg(n=("y", "size"), pe_rate=("y", "mean")).sort_values("n", ascending=False)
viz.bar_counts(axes[3], man["n"], title="scanner (PE rate above bars)", annotate=False)
for i, (n, r) in enumerate(zip(man["n"], man["pe_rate"])):
    axes[3].text(i, n, f"{r:.0%}", ha="center", va="bottom", fontsize=8)
axes[3].tick_params(axis="x", rotation=45)
fig.tight_layout(); viz.save_fig(fig, FIG, NOTEBOOK, "acquisition_variability");

## C. What a clot looks like

### Healthy vs PE, side by side

**What you're looking at.** Top row: PE studies at a positive slice (the middle of their longest positive run). Bottom row: negative studies at the same relative height in the body. Vessel window.

**Why it matters.** This is the whole task in one picture: a human (or a model) has to tell these two rows apart.

**What to look for.** Look inside the bright branching vessels near the center of the chest, next to the heart, for darker gray spots or a vessel that looks cut off. Negative studies show uniformly bright vessels. Do not be discouraged if you cannot see the difference: clots can be a few pixels wide.

In [ ]:
n_pairs = min(4, len(pe_uids), len(neg_uids))
pe_show = sorted(pe_uids, key=lambda u: -metas[u]["n_pos_slices"])[:n_pairs]
images, titles = [], []
rel = []
for u in pe_show:
    v, m = store.load_volume(u); z = labels.representative_positive_slice(m["slice_labels"])
    images.append(v[z]); titles.append(f"PE  {u}  z={z}"); rel.append(z / v.shape[0])
for u, r in zip(neg_uids[:n_pairs], rel):
    v, m = store.load_volume(u); z = int(r * (v.shape[0] - 1))
    images.append(v[z]); titles.append(f"negative  {u}  z={z}")
fig = viz.image_grid(images, titles, ncols=n_pairs, window="vessel")
viz.save_fig(fig, FIG, NOTEBOOK, "healthy_vs_pe");

### Zoom on the central chest

**What you're looking at.** The same PE slices as above, cropped to the central region where the main lung arteries branch off from the heart, shown larger. Vessel window.

**Why it matters.** Clots are small; a full 512-pixel slice hides them. At this zoom the dark spot inside a bright artery is what a radiologist calls a filling defect.

**What to look for.** Bright tubes (arteries full of dye) with a gray or dark patch inside, or a bright tube that ends abruptly. Compare against the uniformly bright vessels in the healthy row above.

In [ ]:
crops = [viz.crop_center(img, frac=0.45) for img in images[:n_pairs]]
fig = viz.image_grid(crops, titles[:n_pairs], ncols=n_pairs, window="vessel", figsize_per=4.2)
viz.save_fig(fig, FIG, NOTEBOOK, "pe_zoomed_center");

### A clot entering and leaving the frame

**What you're looking at.** Consecutive axial slices of one PE study, from a few slices above the labelled clot to a few below. Titles mark which slices are labelled positive. Central crop, vessel window.

**Why it matters.** Shows that a clot is a 3D object: it appears, grows, shrinks and disappears along the z-axis exactly like an object entering and leaving a video. It also visually confirms that the slice labels were aligned to the volume correctly, which the heatmap analysis later depends on.

**What to look for.** The dark spot inside the artery should be present on the slices marked positive and absent just outside that range. If the labelled range and the visible clot disagree, the label alignment is wrong.

In [ ]:
band = labels.positive_band(demo_meta["slice_labels"])
lo, hi = max(band[0] - 3, 0), min(band[1] + 3, demo_vol.shape[0])
zs = np.unique(np.linspace(lo, hi - 1, min(8, hi - lo)).astype(int))
strip = [viz.crop_center(demo_vol[z], frac=0.45) for z in zs]
titles = [f"z={z}  {'PE' if demo_meta['slice_labels'][z] else '-'}" for z in zs]
fig = viz.image_grid(strip, titles, ncols=len(zs), window="vessel", figsize_per=2.6, suptitle=f"study {demo_uid}")
viz.save_fig(fig, FIG, NOTEBOOK, "clot_sequence");

### Central vs peripheral PE

**What you're looking at.** Left: a study labelled *central PE* (clot in the main arteries right next to the heart). Right: a PE study without the central flag (clot further out in smaller branches). Central crop at a positive slice, vessel window.

**Why it matters.** These are the easy and the hard end of the problem. Central clots are big and sit in big vessels; peripheral clots are small objects in thin vessels and are the ones a coarse whole-scan model is expected to miss.

**What to look for.** Left: a large dark region inside a wide bright artery. Right: at most a small gray dot in a thin vessel, possibly not visible at all at this resolution.

In [ ]:
central = [u for u in pe_uids if metas[u]["central_pe"] == 1]
peripheral = [u for u in pe_uids if metas[u]["central_pe"] == 0]
pair, titles = [], []
for name, group in [("central PE", central), ("peripheral PE", peripheral)]:
    if group:
        u = max(group, key=lambda u: metas[u]["n_pos_slices"]); v, m = store.load_volume(u)
        z = labels.representative_positive_slice(m["slice_labels"])
        pair.append(viz.crop_center(v[z], frac=0.45)); titles.append(f"{name}  {u}  z={z}")
if pair:
    fig = viz.image_grid(pair, titles, ncols=2, window="vessel", figsize_per=5)
    viz.save_fig(fig, FIG, NOTEBOOK, "central_vs_peripheral")

### Look-alikes

**What you're looking at.** Studies in the EDA subset flagged *flow artifact* or *true filling defect, not PE*, shown at a mid-chest slice, central crop, vessel window. The caption says which flag applies and whether the study is PE-positive.

**Why it matters.** These are the cases that fool humans: dye mixing unevenly, or something that blocks the dye but is not a clot. A model that keys on 'dark spot in bright vessel' will produce false positives on exactly these. They are the natural hard negatives for a later training stage.

**What to look for.** Dark patches inside vessels that are not clots. If the subset has no such study, the cell says so.

In [ ]:
flagged = [u for u in eda_uids if metas[u]["flow_artifact"] or metas[u]["true_filling_defect_not_pe"]]
if flagged:
    imgs, titles = [], []
    for u in flagged[:8]:
        v, m = store.load_volume(u); z = labels.representative_positive_slice(m["slice_labels"]) or v.shape[0] // 2
        flags = [f for f in ["flow_artifact", "true_filling_defect_not_pe"] if m[f]]
        imgs.append(viz.crop_center(v[z], frac=0.45)); titles.append(f"{u} y={m['y']}\n{', '.join(flags)}")
    fig = viz.image_grid(imgs, titles, ncols=4, window="vessel")
    viz.save_fig(fig, FIG, NOTEBOOK, "look_alikes")
else:
    print("no look-alike flags in the EDA subset")

## D. Where along the body clots appear

The slice labels tell us which slices are positive, but the label file's row order does not
follow slice position (checked on real studies), so positions can only be computed for
studies whose volumes we have assembled: the EDA subset. The embedding notebook records the
aligned labels for every study it processes, so this analysis can be repeated on thousands
of studies later.

### Where clots sit along the head-to-feet axis

**What you're looking at.** Histogram of the position of every PE-positive slice, normalised so 0 is the top of the scan and 1 the bottom, for the EDA PE studies; central and non-central studies overlaid.

**Why it matters.** Tells us which part of the volume the model must attend to, and gives the ground truth for the 'did it look in the right place' check in the heatmap notebook.

**What to look for.** A single hump somewhere in the middle (the lung arteries sit around the heart), central clots more concentrated than peripheral ones, which spread out towards the lung bases.

In [ ]:
pos_central, pos_other = [], []
for u in pe_uids:
    lab = np.asarray(metas[u]["slice_labels"]); z = np.where(lab == 1)[0]
    (pos_central if metas[u]["central_pe"] else pos_other).extend(volume.normalized_z(z, len(lab)).tolist())
fig, ax = plt.subplots(figsize=(8, 3.8))
for pos, color, name in [(pos_other, viz.PALETTE[0], "non-central PE"), (pos_central, viz.PALETTE[3], "central PE")]:
    if pos:
        ax.hist(pos, bins=40, range=(0, 1), alpha=0.6, color=color, label=f"{name} ({len(pos)} slices)", density=True)
ax.set_xlabel("position along the scan (0 = top, 1 = bottom)"); ax.set_ylabel("density"); ax.legend()
viz.save_fig(fig, FIG, NOTEBOOK, "clot_position_distribution");

### Barcode of clot extent

**What you're looking at.** One row per EDA PE study (sorted by number of positive slices), columns are the normalised position along the scan; a cell is red where that slice is positive.

**Why it matters.** One picture showing both how much of each scan is affected and where. It is the visual form of the 'positive slices per study' histogram, with position added.

**What to look for.** Mostly contiguous red blocks (one clot region) of very different lengths, all roughly at the same height; scattered or multiple blocks indicate several clots.

In [ ]:
from matplotlib.colors import ListedColormap

n_bins = 200
order = sorted(pe_uids, key=lambda u: metas[u]["n_pos_slices"])
barcode = np.zeros((len(order), n_bins))
for i, u in enumerate(order):
    lab = np.asarray(metas[u]["slice_labels"]); idx = (volume.normalized_z(np.arange(len(lab)), len(lab)) * (n_bins - 1)).astype(int)
    np.maximum.at(barcode[i], idx, lab)
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(barcode, aspect="auto", cmap=ListedColormap(["white", viz.COLOR_PE]), interpolation="nearest")
ax.set_xlabel("position along the scan (0 = top, 1 = bottom)"); ax.set_ylabel("PE studies, sorted by clot extent")
ax.set_xticks([0, n_bins // 2, n_bins - 1]); ax.set_xticklabels(["0", "0.5", "1"])
viz.save_fig(fig, FIG, NOTEBOOK, "clot_barcode");

## What we learned

    - (fill in after running) how visible the clots were to you at full resolution vs zoomed in.
- (fill in after running) whether the labelled positive range matched the visible clot in the sequence strip (label alignment check).
- (fill in after running) the spread of slice spacing and in-plane resolution, and whether any scanner (kernel) is a PE-rate outlier.
- (fill in after running) where along the scan the clots concentrate and how central differs from peripheral.
- The label file's row order does not follow slice position, so z-position analyses need assembled volumes.
- The test pool is frozen here and is only touched once, at the end of the fine-tuning notebook.